In [ ]:
!pip install -q kagglehub

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")

TensorFlow version: 2.20.0
Libraries imported successfully!


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "uraninjo/augmented-alzheimer-mri-dataset"
)

print("Dataset downloaded successfully!")
print("Dataset path:", dataset_path)

Using Colab cache for faster access to the 'augmented-alzheimer-mri-dataset' dataset.
Dataset downloaded successfully!
Dataset path: /kaggle/input/augmented-alzheimer-mri-dataset


In [ ]:
print("Files and folders inside dataset:")

for item in os.listdir(dataset_path):
    print(item)

Files and folders inside dataset:
OriginalDataset
AugmentedAlzheimerDataset


In [ ]:
for root, dirs, files in os.walk(dataset_path):
    print(root)
    if len(files) > 0:
        print("  Number of images:", len(files))

/kaggle/input/augmented-alzheimer-mri-dataset
/kaggle/input/augmented-alzheimer-mri-dataset/OriginalDataset
/kaggle/input/augmented-alzheimer-mri-dataset/OriginalDataset/ModerateDemented
  Number of images: 64
/kaggle/input/augmented-alzheimer-mri-dataset/OriginalDataset/NonDemented
  Number of images: 3200
/kaggle/input/augmented-alzheimer-mri-dataset/OriginalDataset/VeryMildDemented
  Number of images: 2240
/kaggle/input/augmented-alzheimer-mri-dataset/OriginalDataset/MildDemented
  Number of images: 896
/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset
/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset/ModerateDemented
  Number of images: 6464
/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset/NonDemented
  Number of images: 9600
/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset/VeryMildDemented
  Number of images: 8960
/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset/MildDem

In [ ]:
import os
import shutil
import random

# Original augmented dataset path
source_path = os.path.join(
    dataset_path,
    "AugmentedAlzheimerDataset"
)

# New balanced dataset path
balanced_path = "/content/Alzheimer_Balanced"

# Class names
classes = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]

# Number of images required from each class
IMAGES_PER_CLASS = 6000

# Create balanced dataset folders
for class_name in classes:
    os.makedirs(
        os.path.join(balanced_path, class_name),
        exist_ok=True
    )

# Make selection reproducible
random.seed(42)

# Select and copy images
for class_name in classes:

    class_source = os.path.join(
        source_path,
        class_name
    )

    class_destination = os.path.join(
        balanced_path,
        class_name
    )

    # Get image files
    images = [
        file for file in os.listdir(class_source)
        if file.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(class_name, "available:", len(images))

    # Randomly select 6000 images
    selected_images = random.sample(
        images,
        IMAGES_PER_CLASS
    )

    # Copy selected images
    for i, image_file in enumerate(selected_images):

        source_file = os.path.join(
            class_source,
            image_file
        )

        destination_file = os.path.join(
            class_destination,
            image_file
        )

        shutil.copy2(
            source_file,
            destination_file
        )

    print(class_name, "selected:", len(selected_images))

print("\nBalanced dataset created successfully!")

MildDemented available: 8960
MildDemented selected: 6000
ModerateDemented available: 6464
ModerateDemented selected: 6000
NonDemented available: 9600
NonDemented selected: 6000
VeryMildDemented available: 8960
VeryMildDemented selected: 6000

Balanced dataset created successfully!


In [ ]:
for class_name in classes:

    class_path = os.path.join(
        balanced_path,
        class_name
    )

    count = len([
        file for file in os.listdir(class_path)
        if file.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    print(class_name, ":", count)

MildDemented : 6000
ModerateDemented : 6000
NonDemented : 6000
VeryMildDemented : 6000


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image size and batch size
IMG_SIZE = 224
BATCH_SIZE = 32

# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

# Validation data generator
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Training generator
train_generator = train_datagen.flow_from_directory(
    balanced_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation generator
val_generator = val_datagen.flow_from_directory(
    balanced_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("Training images:", train_generator.samples)
print("Validation images:", val_generator.samples)

Found 19200 images belonging to 4 classes.
Found 4800 images belonging to 4 classes.
Training images: 19200
Validation images: 4800


In [ ]:
class_indices = train_generator.class_indices

class_names = list(class_indices.keys())

print("Class indices:", class_indices)
print("Class names:", class_names)

Class indices: {'MildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'VeryMildDemented': 3}
Class names: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']


In [ ]:
model = keras.Sequential([

    # First convolution block
    layers.Conv2D(
        32,
        (3, 3),
        activation='relu',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    ),
    layers.MaxPooling2D((2, 2)),

    # Second convolution block
    layers.Conv2D(
        64,
        (3, 3),
        activation='relu'
    ),
    layers.MaxPooling2D((2, 2)),

    # Third convolution block
    layers.Conv2D(
        128,
        (3, 3),
        activation='relu'
    ),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        256,
        (3, 3),
        activation='relu'
    ),
    layers.MaxPooling2D((2, 2)),

    # Convert feature maps into one-dimensional vector
    layers.Flatten(),

    # Fully connected layer
    layers.Dense(
        128,
        activation='relu'
    ),

    # Dropout to reduce overfitting
    layers.Dropout(0.5),

    # 4-class output
    layers.Dense(
        4,
        activation='softmax'
    )
])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,718,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,107,652 (19.48 MB)

 Trainable params: 5,107,652 (19.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

Model compiled successfully!


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=7
)

Epoch 1/7
600/600 ━━━━━━━━━━━━━━━━━━━━ 249s 401ms/step - accuracy: 0.2539 - loss: 1.3889 - val_accuracy: 0.2500 - val_loss: 1.3779
Epoch 2/7
600/600 ━━━━━━━━━━━━━━━━━━━━ 240s 401ms/step - accuracy: 0.3564 - loss: 1.3104 - val_accuracy: 0.4948 - val_loss: 1.1152
Epoch 3/7
600/600 ━━━━━━━━━━━━━━━━━━━━ 242s 403ms/step - accuracy: 0.5005 - loss: 1.0852 - val_accuracy: 0.6429 - val_loss: 0.7794
Epoch 4/7
600/600 ━━━━━━━━━━━━━━━━━━━━ 241s 401ms/step - accuracy: 0.5968 - loss: 0.8682 - val_accuracy: 0.6802 - val_loss: 0.6763
Epoch 5/7
600/600 ━━━━━━━━━━━━━━━━━━━━ 238s 397ms/step - accuracy: 0.6299 - loss: 0.7847 - val_accuracy: 0.6910 - val_loss: 0.6493
Epoch 6/7
600/600 ━━━━━━━━━━━━━━━━━━━━ 237s 395ms/step - accuracy: 0.6514 - loss: 0.7400 - val_accuracy: 0.7000 - val_loss: 0.6290
Epoch 7/7
600/600 ━━━━━━━━━━━━━━━━━━━━ 238s 396ms/step - accuracy: 0.6620 - loss: 0.7059 - val_accuracy: 0.6956 - val_loss: 0.6218


In [ ]:
'''history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)'''

'history1 = model.fit(\n    train_generator,\n    validation_data=val_generator,\n    epochs=10\n)'